# Bước 1: Khai báo thư viện và Đọc dữ liệu
Đọc file dữ liệu bổ sung từ thư mục `data/clean/` và loại bỏ các dòng trống rác để giữ nguyên 2.170 dòng thực tế.

In [1]:
import pandas as pd

# 1. Đọc dữ liệu thô
df = pd.read_csv("../data/clean/VNU_co_mo_phong_bo_sung.csv")

# 2. Lọc bỏ dòng trống hoàn toàn dựa trên cột định danh (STT)
df = df.dropna(subset=['STT'])

print(f"Số dòng chuẩn sau khi lọc: {len(df)}")

Số dòng chuẩn sau khi lọc: 2770


# Bước 2: Đồng bộ tên cột và Đánh dấu dữ liệu
- Đổi tên các cột bị sai chính tả theo đúng chuẩn từ điển dữ liệu.
- Đánh dấu 226 dòng trùng lặp (dùng cột `Trung_lap`, bỏ qua cột `STT` khi so sánh) và đánh dấu 442 dòng trả lời đều (dùng cột `Tra_loi_deu`). **Tuyệt đối không xóa dữ liệu gốc.**

In [2]:
# 1. Đổi tên 3 cột sai chính tả
df = df.rename(columns={
    'Time_SocicalMedia': 'Time_SocialMedia',
    'Facilitie_Uni': 'Facilities_Uni',
    'InfuenceF_Friends': 'Influence_Friends'
})

# 2. Đánh dấu dòng trùng lặp (không xóa)
cot_kiem_tra = df.columns.drop('STT')
df['Trung_lap'] = df.duplicated(subset=cot_kiem_tra)
print(f"Số dòng bị đánh dấu trùng lặp: {df['Trung_lap'].sum()}")

# 3. Đánh dấu dòng trả lời đều (chọn 1 mức cho cả 9 câu Likert)
likert_cols = [
    'Adapt_Learning_Uni', 'Study_Methods', 'SupportOf_Uni', 
    'SupportOf_Lec', 'Facilities_Uni', 'Quality_Lecturer', 
    'TrainingCurriculum', 'Competitive_Class', 'Influence_Friends'
]
df['Tra_loi_deu'] = df[likert_cols].nunique(axis=1) == 1
print(f"Số dòng trả lời đều (bừa): {df['Tra_loi_deu'].sum()}")

Số dòng bị đánh dấu trùng lặp: 231
Số dòng trả lời đều (bừa): 557


# Bước 3: Giải mã dữ liệu phân loại (Mapping)
Sử dụng Codebook để chuyển đổi các giá trị số thành nhãn văn bản tiếng Việt dễ đọc, phục vụ cho việc làm Dashboard sau này.

In [3]:
# 1. Giải mã Giới tính
df['Gender'] = df['Gender'].map({1: 'Nam', 2: 'Nữ'})

# 2. Giải mã 9 câu Likert (1-5) thành nhãn chữ
thang_do_5 = {
    1: 'Rất ít', 
    2: 'Ít', 
    3: 'Bình thường', 
    4: 'Nhiều', 
    5: 'Rất nhiều'
}

for col in likert_cols:
    df[col] = df[col].map(thang_do_5)

print("Đã giải mã thành công các biến phân loại!")

Đã giải mã thành công các biến phân loại!


# Bước 4: Tạo các cột phân nhóm mới
- Tạo cột `Nhom_GPA` (chia theo các mức Yếu, Trung bình, Khá, Giỏi, Xuất sắc dựa trên mã hóa 1-5).
- Tạo cột `Nhom_TuHoc` dựa trên thời gian tự học hằng ngày (`Time_Studying`).

In [4]:
# 1. Phân nhóm GPA dựa trên mã 1-5 từ Codebook
df['Nhom_GPA'] = df['GPA'].map({
    1: 'Yếu', 
    2: 'Trung bình', 
    3: 'Khá', 
    4: 'Giỏi', 
    5: 'Xuất sắc'
})

# 2. Phân nhóm Thời gian tự học (Time_Studying)
def phan_nhom_tuhoc(muc_do):
    if muc_do in [1, 2, 3]: return 'mức 1-3'
    elif muc_do == 4: return 'mức 4'
    elif muc_do == 5: return 'mức 5'
    return muc_do

df['Nhom_TuHoc'] = df['Time_Studying'].apply(phan_nhom_tuhoc)

print("Đã tạo xong các nhóm phân tích mới!")

Đã tạo xong các nhóm phân tích mới!


# Bước 5: Kiểm tra toàn vẹn và Xuất file
Kiểm tra lại tổng số dòng dữ liệu để đảm bảo không bị thất thoát, sau đó xuất ra file `vnu_that_sach.csv` mới để cả nhóm sử dụng làm báo cáo và Dashboard.

In [5]:
# Kiểm tra tổng số dòng cuối cùng
print(f"Tổng số dòng dữ liệu sạch cuối cùng: {len(df)}")

# Xuất file kết quả vào thư mục data/clean/ (không lưu kèm cột index thừa)
df.to_csv("../data/clean/vnu_that_sach.csv", index=False)

print("Đã xuất file ../data/clean/vnu_that_sach.csv thành công tuyệt đối!")

Tổng số dòng dữ liệu sạch cuối cùng: 2770
Đã xuất file ../data/clean/vnu_that_sach.csv thành công tuyệt đối!
